# Anexo académico — enunciado original de `lab180`

Este notebook reproduce, **literalmente y en español**, las 9 preguntas del
enunciado académico original sobre `Lab1_engineering_failures.csv`: el
ejercicio de laboratorio (típicamente resuelto con Weka o un árbol de
decisión simple) que dio origen a este dataset.

**Deliberadamente usa el CSV crudo** (180 filas, sin pasar por el contrato de
datos de `schema.py` ni por el auditor de plausibilidad) y un único árbol de
decisión sin restricciones sobre un único split entrenamiento/test — así es
como se plantea el enunciado original, antes de que este repositorio añadiera
validación cruzada repetida, calibración y umbral por coste.

Cada respuesta numérica se calcula en la celda correspondiente, nunca se
escribe a mano — reutilizando las funciones de
`src/predictive_maintenance/eda.py` ya verificadas contra CLAUDE.md §6.

⚠️ **Los datos son SIMULADOS de laboratorio** (CLAUDE.md §7): la celda final
de este notebook explica por qué eso importa para interpretar todo lo que
sigue.

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

from predictive_maintenance import datasets, eda

pd.set_option("display.width", 100)
pd.set_option("display.max_columns", 20)

df = datasets.load("lab180")
target = "failure"
X = df.drop(columns=[target])
y = df[target].astype(str)
print(
    f"Dataset cargado: data/raw/lab180/Lab1_engineering_failures.csv "
    f"-> {df.shape[0]} filas x {df.shape[1]} columnas"
)

Dataset cargado: data/raw/lab180/Lab1_engineering_failures.csv -> 180 filas x 6 columnas


## Pregunta 1 — ¿Cuántas instancias (filas) tiene el dataset?

In [2]:
n_instancias = len(df)
print(f"Número de instancias: {n_instancias}")

Número de instancias: 180


## Pregunta 2 — ¿Cuántos atributos tiene?

In [3]:
atributos = eda.numeric_attributes(df, target)
n_atributos = len(atributos)
print(f"Número de atributos predictivos: {n_atributos} -> {atributos}")
print("(+ 1 variable objetivo: 'failure')")

Número de atributos predictivos: 5 -> ['temperature_c', 'vibration_mm_s', 'pressure_bar', 'hours_since_maintenance', 'load_percent']
(+ 1 variable objetivo: 'failure')


## Pregunta 3 — Para cada atributo: mínimo, máximo, media, desviación típica y valores ausentes

In [4]:
tabla = eda.descriptive_table(df, target)
tabla_mostrada = tabla[["n", "nulos", "min", "max", "media", "desv_tipica"]].round(4)
print(tabla_mostrada.to_string())

                           n  nulos    min     max     media  desv_tipica
atributo                                                                 
temperature_c            176      4  47.00   89.80   67.8102       7.6651
vibration_mm_s           176      4  -0.34    9.59    4.2438       1.3349
pressure_bar             176      4   4.27   10.19    6.7630       1.1330
hours_since_maintenance  180      0  20.00  896.00  473.1889     262.3427
load_percent             180      0  31.50  100.00   71.0900      14.4342


## Pregunta 4 — ¿Es `failure` un atributo nominal con exactamente dos clases? Verificarlo

In [5]:
balance = eda.class_balance(df, target, "yes")
clases = sorted(y.unique())
print(f"dtype de 'failure': {df[target].dtype}")
print(f"Valores únicos: {clases}")
print(f"¿Exactamente dos clases?: {len(clases) == 2}")
print(f"Nulos en 'failure': {balance['nulos_en_objetivo']}")
print(f"Conteos: {balance['conteos']}")

dtype de 'failure': str
Valores únicos: ['no', 'yes']
¿Exactamente dos clases?: True
Nulos en 'failure': 0
Conteos: {'no': 170, 'yes': 10}


## Pregunta 5 — ¿Qué porcentaje de instancias clasifica correctamente el árbol?

Split 70/30 estratificado, semilla 42 (CLAUDE.md §2.10: semillas fijadas, dos
ejecuciones producen resultados idénticos), imputación por mediana (sin
`Pipeline` de por medio: este notebook, a propósito, es el ejercicio
"ingenuo" que el resto del repositorio mejora).

**Regla dura de este repositorio (CLAUDE.md §2.1): la accuracy nunca se
reporta sola — siempre junto a la fila del `DummyClassifier`.**

In [6]:
imputer_medianas = X.median(numeric_only=True)
X_imputado = X.fillna(imputer_medianas)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputado, y, test_size=0.30, stratify=y, random_state=42
)
arbol = DecisionTreeClassifier(random_state=42)
arbol.fit(X_train, y_train)
y_pred = arbol.predict(X_test)
accuracy_arbol = accuracy_score(y_test, y_pred)

accuracy_dummy = balance["accuracy_trivial_mayoritario_pct"]
print(f"Split: {len(X_train)} entrenamiento / {len(X_test)} test (70/30, estratificado, seed=42)")
print(f"% de instancias correctamente clasificadas (árbol, test): {100 * accuracy_arbol:.4f} %")
print(
    f"Accuracy del clasificador trivial mayoritario (SIEMPRE al lado, CLAUDE.md §2.1): "
    f"{accuracy_dummy:.4f} %"
)
if 100 * accuracy_arbol <= accuracy_dummy:
    print(
        "-> El árbol NO supera al clasificador trivial en este split: no aporta nada por sí solo."
    )
else:
    print("-> El árbol supera al trivial, aunque con pocos positivos la comparación es frágil.")

Split: 126 entrenamiento / 54 test (70/30, estratificado, seed=42)
% de instancias correctamente clasificadas (árbol, test): 90.7407 %
Accuracy del clasificador trivial mayoritario (SIEMPRE al lado, CLAUDE.md §2.1): 94.4444 %
-> El árbol NO supera al clasificador trivial en este split: no aporta nada por sí solo.


## Pregunta 6 — Matriz de confusión: ¿cómo se lee?

In [7]:
etiquetas = ["no", "yes"]
matriz = confusion_matrix(y_test, y_pred, labels=etiquetas)
tn, fp, fn, tp = matriz.ravel()
print("Matriz de confusión (filas = real, columnas = predicho; orden ['no', 'yes']):")
print(
    pd.DataFrame(
        matriz,
        index=[f"real={c}" for c in etiquetas],
        columns=[f"predicho={c}" for c in etiquetas],
    )
)
print()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"Positivos reales en el test set: {fn + tp} (de {balance['n_positivos']} totales)")

Matriz de confusión (filas = real, columnas = predicho; orden ['no', 'yes']):
          predicho=no  predicho=yes
real=no            48             3
real=yes            2             1

TN=48  FP=3  FN=2  TP=1
Positivos reales en el test set: 3 (de 10 totales)


## Pregunta 7 — ¿El modelo se equivoca más en fallos o en no-fallos?

In [8]:
print(f"Falsos negativos (fallos reales predichos como 'no'): {fn}")
print(f"Falsos positivos (no-fallos reales predichos como 'yes'): {fp}")
if fn > fp:
    print("-> El modelo se equivoca MÁS prediciendo FALLOS (más falsos negativos que positivos).")
elif fp > fn:
    print(
        "-> El modelo se equivoca MÁS prediciendo NO-FALLOS (más falsos positivos que negativos)."
    )
else:
    print("-> El modelo se equivoca por igual en ambas direcciones en este split concreto.")

Falsos negativos (fallos reales predichos como 'no'): 2
Falsos positivos (no-fallos reales predichos como 'yes'): 3
-> El modelo se equivoca MÁS prediciendo NO-FALLOS (más falsos positivos que negativos).


## Pregunta 8 — ¿Qué atributos aparecen en el árbol entrenado?

In [9]:
importancias = pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
usados = importancias[importancias > 0]
print("Importancia de cada atributo en el árbol entrenado (0 = no aparece en ningún corte):")
print(importancias.round(4).to_string())
print()
print(f"Atributos que SÍ aparecen en el árbol: {list(usados.index)}")
print(f"Profundidad del árbol: {arbol.get_depth()}  |  nº de hojas: {arbol.get_n_leaves()}")

Importancia de cada atributo en el árbol entrenado (0 = no aparece en ningún corte):
hours_since_maintenance    0.4031
load_percent               0.2640
pressure_bar               0.1830
vibration_mm_s             0.1008
temperature_c              0.0490

Atributos que SÍ aparecen en el árbol: ['hours_since_maintenance', 'load_percent', 'pressure_bar', 'vibration_mm_s', 'temperature_c']
Profundidad del árbol: 4  |  nº de hojas: 7


## Pregunta 9 — Interpretación: sentido ingenieril, primer atributo de
división, combinación de condiciones hacia `failure="yes"`, facilidad de
interpretación y ventaja frente a modelos opacos

### Primer atributo de división (raíz del árbol)

In [10]:
idx_raiz = arbol.tree_.feature[0]
atributo_raiz = X.columns[idx_raiz]
umbral_raiz = arbol.tree_.threshold[0]
print(f"Primer atributo de división (ESTE split concreto): '{atributo_raiz}' <= {umbral_raiz:.3f}")
print()
print("ADVERTENCIA (CLAUDE.md §10.1): sobre 300 bootstraps estratificados, la vibración")
print("es la raíz del árbol solo ~51.3% de las veces (56.0% en la ejecución medida de este")
print("repositorio, ver reports/explainability_lab180.json) -- NO en el 100% de los remuestreos.")
print("Un único árbol, como el de esta celda, no es prueba de que 'el atributo que decide")
print("el fallo es la vibración'. tests/test_tree_stability.py protege explícitamente esto.")

Primer atributo de división (ESTE split concreto): 'pressure_bar' <= 5.360

ADVERTENCIA (CLAUDE.md §10.1): sobre 300 bootstraps estratificados, la vibración
es la raíz del árbol solo ~51.3% de las veces (56.0% en la ejecución medida de este
repositorio, ver reports/explainability_lab180.json) -- NO en el 100% de los remuestreos.
Un único árbol, como el de esta celda, no es prueba de que 'el atributo que decide
el fallo es la vibración'. tests/test_tree_stability.py protege explícitamente esto.


### Sentido ingenieril de los atributos (asociación univariante, AUC de Mann-Whitney, CLAUDE.md §6.4)

In [11]:
univariante = eda.univariate_auc(df, target, "yes")
print(univariante[["auc", "p_valor", "direccion", "significativo_5pct"]].round(4).to_string())

                            auc  p_valor direccion  significativo_5pct
atributo                                                              
vibration_mm_s           0.8503   0.0002   directa                True
pressure_bar             0.2515   0.0085   inversa                True
temperature_c            0.7259   0.0167   directa                True
hours_since_maintenance  0.7218   0.0187   directa                True
load_percent             0.6682   0.0746   directa               False


`vibration_mm_s` (AUC 0.8503) es la firma canónica de degradación de
rodamientos y desalineación de ejes: el aumento de amplitud RMS precede al
fallo catastrófico. `temperature_c` (0.7259) y `hours_since_maintenance`
(0.7218) apuntan en la dirección esperada: más fricción/peor refrigeración y
más riesgo acumulado, respectivamente. **`pressure_bar` se asocia
INVERSAMENTE al fallo (AUC 0.2515)** — contraintuitivo si se espera "más
presión, más estrés", pero compatible con pérdida de estanqueidad, fuga o una
bomba perdiendo eficiencia volumétrica: presión BAJA, no alta, precede al
fallo. **`load_percent` NO es significativo** (p = 0.0746, por encima de
0.05): va en la dirección esperada pero no se puede concluir nada con esta
muestra — no se inventa una historia sobre ella.

### Combinación de condiciones que lleva a `failure="yes"`

In [12]:
reglas = export_text(arbol, feature_names=list(X.columns), max_depth=3)
print(reglas)

|--- pressure_bar <= 5.36
|   |--- load_percent <= 84.60
|   |   |--- hours_since_maintenance <= 727.50
|   |   |   |--- class: no
|   |   |--- hours_since_maintenance >  727.50
|   |   |   |--- class: yes
|   |--- load_percent >  84.60
|   |   |--- class: yes
|--- pressure_bar >  5.36
|   |--- hours_since_maintenance <= 894.00
|   |   |--- temperature_c <= 81.70
|   |   |   |--- class: no
|   |   |--- temperature_c >  81.70
|   |   |   |--- vibration_mm_s <= 4.41
|   |   |   |   |--- class: no
|   |   |   |--- vibration_mm_s >  4.41
|   |   |   |   |--- class: yes
|   |--- hours_since_maintenance >  894.00
|   |   |--- class: yes


Una de las rutas hacia `class: yes` en este árbol:
`pressure_bar > 5.36` **y** `hours_since_maintenance > 894.00` predice
directamente un fallo — presión ya normalizada (no es la rama de fuga) pero
con muchísimas horas acumuladas desde el último mantenimiento: la rama
derecha de la curva de bañera (CLAUDE.md §11). Otra ruta,
`pressure_bar <= 5.36` **y** `load_percent > 84.60`, combina presión baja
(posible fuga) con carga alta — dos factores de riesgo independientes
apuntando en la misma dirección.

### Facilidad de interpretación y ventaja frente a modelos opacos

Un árbol de decisión se lee como una secuencia de preguntas sí/no que
cualquier ingeniero de mantenimiento puede seguir sin entender estadística:
"¿la presión es menor que 5.36 bar? ¿la carga supera el 84.6 %?". Esa
trazabilidad es su ventaja frente a un modelo opaco (una red neuronal, un
ensamble grande): se puede auditar CADA decisión, señalar exactamente qué
condición la disparó, y decidir si esa condición tiene sentido físico — como
se hizo arriba con `pressure_bar`. El coste de esa transparencia es la
inestabilidad ya demostrada (Pregunta 9, primer apartado): la MISMA
trazabilidad que hace legible un árbol concreto es la que expone lo poco que
cambia con la muestra — dos virtudes que no se pueden separar.

---

**Estas preguntas asumen clases equilibradas. El pipeline principal de este
repositorio explica por qué la accuracy es la métrica equivocada para este
problema — ver [`README.md`](../README.md).**